### DTSA 5510 Unsupervised Algorithms in Machine Learning - Mini Project (Part 2)
Tsai-Yun Li, 18 Oct 2023

##### Topic: Limitation(s) of sklearn’s non-negative matrix factorization library <br>
Requirements: <br>
Upload a Jupyter notebook with a mix of code/markdown to answer the following questions. Please mark the sections of your notebook as 1 and 2 so that graders can follow along. <br>

1. Load the movie ratings data (as in the HW3-recommender-system) and use matrix factorization technique(s) and predict the missing ratings from the test data. Measure the RMSE. You should use sklearn library. [10 pts]

Make sure that your notebook includes the following: <br>

use's sklearn's non-negative matrix factorization <br>

notebook shows the RMSE with an analysis of what that RMSE means <br>

2. Discuss the results and why they did not work well compared to simple baseline or similarity-based methods we’ve done in Module 3. Can you suggest a way(s) to fix it? [10 pts]

### 1. Loading the Data

In [1]:
import numpy as np
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
import pandas as pd

# loading the movie ratings data (as in the HW3-recommender-system)
MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [2]:
# checking column names in MV_users
MV_users.head()

,uID,gender,age,accupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [3]:
# checking column names in MV_movies
MV_movies.head()

,mID,title,year,Doc,Com,Hor,Adv,Wes,Dra,Ani,...,Chi,Cri,Thr,Sci,Mys,Rom,Fil,Fan,Act,Mus
0,1,Toy Story,1995,0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1,2,Jumanji,1995,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,1,0,0
2,3,Grumpier Old Men,1995,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,1995,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II,1995,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. Creating User-Movie Matrices to Store Ratings

In [4]:
## setting the stage
# getting the number of users and movies
n_users = MV_users['uID'].nunique(); print("n_users =", n_users)
n_movies = MV_movies['mID'].nunique(); print("n_movies =", n_movies)

# creating a mapping for uID and mID to a continuous range
user_mapping = {uid: i for i, uid in enumerate(MV_users['uID'].unique())}
movie_mapping = {mid: i for i, mid in enumerate(MV_movies['mID'].unique())}
#print("user_mapping:", user_mapping) #key 'uid', and value index i from 0 to n_users -1
#print("movie_mapping:", movie_mapping) #key 'mid', and value index i from 0 to n_movie -1

n_users = 6040
n_movies = 3883


##### 2.1 Creating User-Movie Matrices for the Training Dataset

In [5]:
UMmatrix_train = np.zeros((n_users, n_movies))

for line in train.itertuples():
    UMmatrix_train[user_mapping[line.uID], movie_mapping[line.mID]] = line.rating

In [6]:
print("trainset user-movie matrix storing ratings, i.e. X_train", UMmatrix_train) 
#row: user1, user2, ..., userN #column: movie1, movie2, ..., movieN
#user-item, here, user-movie matrix # values in the matrix: ratings each user gave to each movie

trainset user-movie matrix storing ratings, i.e. X_train [[5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [3. 0. 0. ... 0. 0. 0.]]


##### 2.2 Creating User-Movie Matrices for the Test Dataset

In [7]:
UMmatrix_test = np.zeros((n_users, n_movies))

for line in test.itertuples():
    UMmatrix_test[user_mapping[line.uID], movie_mapping[line.mID]] = line.rating

In [8]:
print("testset user-movie matrix storing ratings, i.e. y_test:", UMmatrix_test) 
#row: user1, user2, ..., userN #column: movie1, movie2, ..., movieN
#user-item, here, user-movie matrix # values in the matrix: ratings each user gave to each movie

testset user-movie matrix storing ratings, i.e. y_test: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


### 3. Building and Implenting NMF

In [9]:
from sklearn.decomposition import NMF
X_train = UMmatrix_train

NMF_model = NMF(n_components=10, random_state=2023)
W = NMF_model.fit_transform(X_train)
H = NMF_model.components_

In [10]:
# predicting the ratings from the training dataset
predicted_ratings = np.dot(W, H)
print("predicted the ratings from the training dataset, i.e. X_pred:", predicted_ratings)

predicted the ratings from the training dataset, i.e. X_pred: [[1.80936614e+00 5.84322488e-01 1.82708596e-01 ... 2.20471103e-03
  1.80501860e-03 4.81851590e-02]
 [7.35848032e-01 3.24580298e-01 2.40159397e-01 ... 1.12646401e-02
  1.97655203e-03 9.56772277e-02]
 [8.65964928e-01 1.45984911e-01 9.73919165e-02 ... 3.17703102e-03
  2.32015088e-03 2.91416639e-02]
 ...
 [2.40597336e-01 4.96260734e-02 2.99208931e-02 ... 4.35050668e-03
  2.81969018e-03 1.99664577e-02]
 [1.17868115e+00 4.24708724e-01 1.04499933e-01 ... 4.12703881e-02
  2.35486467e-03 8.31754378e-03]
 [1.01565816e+00 4.43137430e-02 5.69562602e-02 ... 1.08395822e-01
  7.24993749e-02 5.14529393e-01]]


### 4. Computing the Root Mean Squared Error (RMSE) of NMF

In [11]:
from sklearn.metrics import mean_squared_error
# getting the none-zero ratings in the user-movie matrix from the test dataset
nonezeroR_test = UMmatrix_test.nonzero()
print("none-zero ratings in the user-movie matrix from the test dataset:", nonezeroR_test, "\n")
# indices (0, 604), (0, 907), (0,156), ... (6039, 3436), (6039, 3602), (6039, 3749)

# predicting the ratings for the test dataset, y_pred
y_pred = predicted_ratings[nonezeroR_test]
print("predicted the ratings for the test dataset, i.e. y_pred:", y_pred)

# getting the ground truth ratings for the test dataset, y_test
y_test = UMmatrix_test[nonezeroR_test]

none-zero ratings in the user-movie matrix from the test dataset: (array([   0,    0,    0, ..., 6039, 6039, 6039]), array([ 604,  907, 1506, ..., 3436, 3602, 3749])) 

predicted the ratings for the test dataset, i.e. y_pred: [0.85100614 1.82849578 0.00537186 ... 0.33979948 1.38766475 0.63644295]


In [12]:
# computing the rmse of the NMF prediction on the test dataset
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"The RMSE value is {rmse}, which measures the prediction accuracy of the NMF_model.")
print("It is the summed error between y_test and y_pred.", "\n")

min_rating = UMmatrix_train[UMmatrix_train.nonzero()].min()
max_rating = UMmatrix_train[UMmatrix_train.nonzero()].max()

print("Since the RMSE value is in the same unit as the ratings themselves", 
    f"and the rating range of the movies is from {min_rating} to {max_rating}",
    f"on average the NMF model's predictions are {rmse} rating points off from the actual ratings.")

The RMSE value is 2.9118143081496006, which measures the prediction accuracy of the NMF_model.
It is the summed error between y_test and y_pred. 

Since the RMSE value is in the same unit as the ratings themselves and the rating range of the movies is from 1.0 to 5.0 on average the NMF model's predictions are 2.9118143081496006 rating points off from the actual ratings.


### 5. Discussion:
The NMF model's prediction power is not so great, because it has an RMSE of around 2.9, which is pretty large (movie rating range is 1.0 to 5.0). <br>

The simple basline and the similarity-based methods we've done in Module 3 both have an RMSE of about 1. Thus, compared to these simplier models, the NMF model performs poorly. <br>

Why does this happen? Why do the simplier models work better than the NMF model? How could we fix this? <br>

This could be caused by the fact that the rating matrix is very sparse, with lots of zeros. NMF is poor at capturing underlying patterns of sparse matrices. It also probably overfits to the training dataset. Besides, the movie rating is 1.0 to 5.0, meaning that the zeros stands for missing values. We could try to fix this by converting the sparse rating matrix to a dense one by imputing some number for the missing values, such as the average rating 3.0, before traing the NMF. We can also perform hyperparamter tuning or some regularizations to optimize the performance of NMF.